In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import time

print("Imports done")

Imports done


In [2]:
data_path = r"C:\Users\lenovo\Desktop\Projects\Network-IDS\data"

X_train = pd.read_csv(os.path.join(data_path, "X_train.csv"))
X_test  = pd.read_csv(os.path.join(data_path, "X_test.csv"))
y_train = pd.read_csv(os.path.join(data_path, "y_train.csv")).squeeze()
y_test  = pd.read_csv(os.path.join(data_path, "y_test.csv")).squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts()}")

X_train: (1222540, 80)
X_test:  (305636, 80)
y_train distribution:
Label_Binary
1    676998
0    545542
Name: count, dtype: int64


In [3]:
print("Training Random Forest...")
start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rf_model.fit(X_train, y_train)

elapsed = time.time() - start
print(f"Training complete in {elapsed:.1f} seconds")

Training Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  2.1min


Training complete in 288.1 seconds


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  4.8min finished


In [4]:
rf_preds = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print("=" * 55)
print("   RANDOM FOREST — EVALUATION RESULTS")
print("=" * 55)
print(classification_report(y_test, rf_preds, target_names=["BENIGN", "ATTACK"]))
print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_proba):.4f}")

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    2.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.9s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    2.2s finished


   RANDOM FOREST — EVALUATION RESULTS
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00    136386
      ATTACK       1.00      1.00      1.00    169250

    accuracy                           1.00    305636
   macro avg       1.00      1.00      1.00    305636
weighted avg       1.00      1.00      1.00    305636

ROC-AUC Score: 1.0000


In [5]:
print("Training Logistic Regression...")
start = time.time()

lr_model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
lr_model.fit(X_train, y_train)

elapsed = time.time() - start
print(f"Training complete in {elapsed:.1f} seconds")

Training Logistic Regression...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.


Training complete in 139.1 seconds


In [6]:
lr_preds = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]

print("=" * 55)
print("   LOGISTIC REGRESSION — EVALUATION RESULTS")
print("=" * 55)
print(classification_report(y_test, lr_preds, target_names=["BENIGN", "ATTACK"]))
print(f"ROC-AUC Score: {roc_auc_score(y_test, lr_proba):.4f}")

   LOGISTIC REGRESSION — EVALUATION RESULTS
              precision    recall  f1-score   support

      BENIGN       0.96      0.91      0.94    136386
      ATTACK       0.93      0.97      0.95    169250

    accuracy                           0.95    305636
   macro avg       0.95      0.94      0.95    305636
weighted avg       0.95      0.95      0.95    305636

ROC-AUC Score: 0.9861


In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = {
    "Model": ["Random Forest", "Logistic Regression"],
    "Accuracy":  [accuracy_score(y_test, rf_preds),  accuracy_score(y_test, lr_preds)],
    "Precision": [precision_score(y_test, rf_preds), precision_score(y_test, lr_preds)],
    "Recall":    [recall_score(y_test, rf_preds),    recall_score(y_test, lr_preds)],
    "F1-Score":  [f1_score(y_test, rf_preds),        f1_score(y_test, lr_preds)],
    "ROC-AUC":   [roc_auc_score(y_test, rf_proba),   roc_auc_score(y_test, lr_proba)]
}

results_df = pd.DataFrame(results)
results_df = results_df.set_index("Model")
results_df = results_df.round(4)
print(results_df.to_string())

                     Accuracy  Precision  Recall  F1-Score  ROC-AUC
Model                                                              
Random Forest          0.9993     0.9991  0.9997    0.9994   1.0000
Logistic Regression    0.9471     0.9340  0.9733    0.9532   0.9861


In [8]:
models_path = r"C:\Users\lenovo\Desktop\Projects\Network-IDS\models"

with open(os.path.join(models_path, "random_forest.pkl"), "wb") as f:
    pickle.dump(rf_model, f)

with open(os.path.join(models_path, "logistic_regression.pkl"), "wb") as f:
    pickle.dump(lr_model, f)

print("Models saved successfully")
print(f"\nmodels/ folder now contains:")
for f in os.listdir(models_path):
    print(f"  {f}")

Models saved successfully

models/ folder now contains:
  label_encoder.pkl
  logistic_regression.pkl
  random_forest.pkl
  scaler.pkl
